# Multi-Layer Equity Network Analysis

Adapted from `ycs_multilayer.ipynb`. Each layer is an equity index (CAC40, DAX30,
FTSE100, HSI50, MIB40; SP500 excluded by default for performance -- see
`STOCK_FILTER`/`EXCLUDE_INDICES` in Chapter A). Nodes are stock tickers. Correlations
are computed on **daily returns**, not raw price levels, via
`tgraphportfolio.analysis.transforms.apply_transforms(..., ["daily_returns"])`.

## Key Concepts
- **Intra-layer edges**: Correlations between stocks within the same equity index
- **Inter-layer edges**: Direct connections linking the same stock across indices
  (structurally near-empty for this dataset -- see Chapter C overlap check)
- **Temporal evolution**: Rolling window networks over time for each layer + aggregate
  multi-layer structure
- **Multi-layer metrics**: Degree centrality, density, components accounting for all layers

## Part II note
Part II adapts the source notebook's Nelson-Siegel factor/residual-network methodology
to equities via a statistical PCA factor model per index (no Nelson-Siegel curve fit --
there is no maturity axis for equity indices), since Nelson-Siegel itself has no equity
analogue. `PC1` plays the role of a market/index factor, analogous to level; residual
networks isolate idiosyncratic (stock-specific) co-movement from that common factor.

## Contents
**Part I: Multi-Layer Network Evolution**
1. Chapter A -- Data Loading, Returns Transform, and Exploration
2. Chapter B -- Single-Layer (per-EqIndex) Network Construction
3. Chapter C -- Multi-Layer Network Structure
4. Chapter D -- Multi-Layer Network Metrics
5. Chapter E -- Temporal Analysis of Multi-Layer Evolution
6. Chapter F -- Inter-Layer Connectivity Analysis
7. Chapter G -- Community Detection in Multi-Layer Networks

**Part II: Factor Models, Residuals, and Stress**
8. Chapter H -- Statistical Factor Extraction (PCA per EqIndex)
9. Chapter I -- PCA-Residual Networks
10. Chapter J -- Factor Evolution & Regime Classification
11. Chapter K -- Stress Testing & Correlation Breakdown
12. Chapter L -- Advanced Centrality & Network Influence
13. Chapter M -- 3D Interactive Visualization
14. Chapter N -- Signal Subgraph Detection
15. Chapter O -- Multi-Layer Community Detection with Layer Awareness

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import itertools
import numpy as np
import polars as pl
import networkx as nx
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
from plotnine import *

from tgraphportfolio.analysis import transforms
from tgraphportfolio.analysis.measures import compute_measure, available_measures
from tgraphportfolio.analysis.network import build_corr_nx, pivot_to_wide
from tgraphportfolio.analysis.pyvis_plot import multilayer_graph_to_html
from tgraphportfolio.analysis.yield_curve_factors import (
    fit_pca, classify_yield_curve_regimes, compute_node_factor_importance,
)
from tgraphportfolio.backends.duckdb_backend import DuckDBSource

from graspologic.embed import AdjacencySpectralEmbed
from sklearn.cluster import KMeans
from multi_layer_NetViz_fcts import Layer, LayeredNetworkGraph

print(f"Imports successful at {datetime.now()}")

# Part I: Multi-Layer Network Evolution

## Chapter A: Data Loading, Returns Transform, and Exploration

Load daily EOD prices from DuckDB, transform `Close` to daily returns, and explore
the panel's dimensions, missingness, and per-index coverage.

In [ ]:
DUCKDB_PATH = r"D:\data\duckdb\equity_eod_data.duckdb"
TABLE = "equity_eod"
DATE_COL, NAME_COL, VALUE_COL, LAYER_COL = "Date", "Stock", "Close", "EqIndex"

# Filtering config:
# - EXCLUDE_INDICES: applied only when STOCK_FILTER is None (default: skip SP500 for
#   rolling-window performance -- ~505 tickers vs 30-100 for the other five indices;
#   see Chapter B for the performance estimate).
# - STOCK_FILTER: optional list of exact Stock tickers to include, applied across ALL
#   EqIndex layers -- overrides EXCLUDE_INDICES, so a curated subset of SP500 names can
#   be re-included without pulling in the full index.
EXCLUDE_INDICES: list[str] | None = ["SP500"]
STOCK_FILTER: list[str] | None = None  # e.g. ["AAPL", "MSFT", "JPM"]

with DuckDBSource(DUCKDB_PATH, read_only=True) as db:
    df_raw = db.run_query(f"""
        SELECT CAST("Index" AS DATE) AS {DATE_COL}, Stock AS {NAME_COL},
               Close AS {VALUE_COL}, EqIndex AS {LAYER_COL}
        FROM {TABLE}
        ORDER BY {DATE_COL}, {LAYER_COL}, {NAME_COL}
    """)

if STOCK_FILTER:
    df_raw = df_raw.filter(pl.col(NAME_COL).is_in(STOCK_FILTER))
elif EXCLUDE_INDICES:
    df_raw = df_raw.filter(~pl.col(LAYER_COL).is_in(EXCLUDE_INDICES))

print(f"Loaded {df_raw.height:,} rows across {df_raw.select(LAYER_COL).n_unique()} indices")
print(df_raw.schema)
df_raw.head(10)

In [ ]:
df_returns = transforms.apply_transforms(
    df_raw,
    transform_ids=["daily_returns"],
    date_column=DATE_COL,
    name_column=NAME_COL,      # per-stock, not per-EqIndex -- returns are a per-security concept
    value_columns=[VALUE_COL],
)
print(f"Rows before transform: {df_raw.height:,}")
print(f"Rows after daily_returns (first obs/stock dropped): {df_returns.height:,}")
df_returns.head(10)

In [ ]:
df_clean = df_returns.drop_nulls(subset=[DATE_COL, NAME_COL, VALUE_COL, LAYER_COL])

dates_unique = sorted(df_clean.select(DATE_COL).unique()[DATE_COL].to_list())
eqindex_unique = sorted(df_clean.select(LAYER_COL).unique()[LAYER_COL].to_list())
stocks_unique = sorted(df_clean.select(NAME_COL).unique()[NAME_COL].to_list())

print(f"Date range: {dates_unique[0]} to {dates_unique[-1]} ({len(dates_unique)} trading days)")
print(f"Equity indices (layers): {eqindex_unique}")
print(f"Total stocks (nodes): {len(stocks_unique)}")

In [ ]:
# Stocks per index and observation counts -- expect ~30-100 stocks per index (SP500 excluded)
coverage_df = (
    df_clean.group_by(LAYER_COL)
    .agg(
        pl.col(NAME_COL).n_unique().alias("n_stocks"),
        pl.col(DATE_COL).n_unique().alias("n_dates"),
        pl.len().alias("n_obs"),
    )
    .sort(LAYER_COL)
)
coverage_df

In [ ]:
# Missingness: how many (date, stock) combinations are absent from a fully dense panel
missing_summary = []
for eq_index in eqindex_unique:
    sub = df_clean.filter(pl.col(LAYER_COL) == eq_index)
    n_stocks = sub.select(NAME_COL).n_unique()
    n_dates = sub.select(DATE_COL).n_unique()
    dense_expected = n_stocks * n_dates
    actual = sub.height
    missing_summary.append({
        LAYER_COL: eq_index,
        "expected_dense": dense_expected,
        "actual_obs": actual,
        "pct_present": round(100 * actual / dense_expected, 2) if dense_expected else 0.0,
    })
pl.DataFrame(missing_summary)

In [ ]:
coverage_plot = (
    ggplot(coverage_df.to_pandas(), aes(x=LAYER_COL, y="n_stocks"))
    + geom_col(fill="#0ea5e9")
    + labs(title="Stocks per Equity Index (Layer)", x="EqIndex", y="Number of Stocks")
    + theme_minimal()
)
coverage_plot

## Chapter B: Single-Layer (per-EqIndex) Network Construction

Build a rolling-window correlation network per equity index. Nodes are stock tickers;
edges are Spearman correlations of daily returns within that index and window.

In [ ]:
def build_index_network(
    df: pl.DataFrame,
    eq_index: str,
    date_start,
    date_end,
    measure: str = 'spearman_correlation',
    independent_threshold: float = 0.33,
) -> nx.Graph:
    """Single-layer network for one EqIndex/window. Nodes=Stock, edges=return correlation."""
    df_layer = df.filter(
        (pl.col(LAYER_COL) == eq_index)
        & (pl.col(DATE_COL) >= date_start)
        & (pl.col(DATE_COL) <= date_end)
    )
    if df_layer.height < 3:
        return nx.Graph()

    df_wide = pivot_to_wide(df_layer, date_column=DATE_COL, name_column=NAME_COL, value_column=VALUE_COL)
    nodes = [c for c in df_wide.columns if c != DATE_COL]
    if len(nodes) < 2:
        return nx.Graph()

    measure_df = compute_measure(measure, df_wide.select(nodes), nodes, progress=None)
    return build_corr_nx(measure_df, independent_threshold=independent_threshold)

In [ ]:
window_size = 252  # ~1 trading year -- matches dax_network_evolution.ipynb's equity-tuned default
step_size = 21       # ~1 trading month

windows = [
    (dates_unique[i], dates_unique[i + window_size - 1])
    for i in range(0, len(dates_unique) - window_size + 1, step_size)
]
window_indices = list(range(len(windows)))

print(f"{len(windows)} windows, {windows[0]} .. {windows[-1]}")

In [ ]:
# Performance note: cost per window scales O(n_stocks^2) per index. With SP500 excluded
# by default, the largest remaining index (FTSE100, ~99 stocks) costs ~4,851 pairs/window;
# across all 5 indices and ~139 windows this is on the order of a few hundred thousand
# pairwise Spearman calls -- a few minutes, not hours. Re-including SP500 (~505 stocks,
# ~127K pairs/window) would push this to an estimated multi-hour run; use STOCK_FILTER
# in Chapter A to cherry-pick specific large-cap names instead of the full index.
layer_networks: dict[str, list[nx.Graph]] = {eq_index: [] for eq_index in eqindex_unique}
layer_metadata: dict[str, list[dict]] = {eq_index: [] for eq_index in eqindex_unique}

for eq_index in eqindex_unique:
    for w_idx, (date_start, date_end) in enumerate(tqdm(windows, desc=eq_index)):
        G = build_index_network(df_clean, eq_index, date_start, date_end)
        layer_networks[eq_index].append(G)
        layer_metadata[eq_index].append({
            "window_idx": w_idx,
            "date_start": date_start,
            "date_end": date_end,
            "n_nodes": G.number_of_nodes(),
            "n_edges": G.number_of_edges(),
        })

print("Done building per-index rolling networks.")

In [ ]:
for eq_index in eqindex_unique:
    n_edges_avg = np.mean([m["n_edges"] for m in layer_metadata[eq_index]])
    n_nodes_avg = np.mean([m["n_nodes"] for m in layer_metadata[eq_index]])
    print(f"{eq_index}: avg {n_nodes_avg:.0f} nodes, {n_edges_avg:.1f} edges/window")

## Chapter C: Multi-Layer Network Structure

Assemble the per-index layer graphs into a single multiplex network per window. Node
IDs are `(stock, eq_index)` tuples. Intra-layer edges come from each index's
correlation graph; inter-layer edges would connect the same stock across indices --
but see the overlap check below.

In [ ]:
# Confirm cross-index ticker overlap before assuming inter-layer edges will exist.
overlap = (
    df_clean.select([LAYER_COL, NAME_COL]).unique()
    .group_by(NAME_COL).agg(pl.col(LAYER_COL).n_unique().alias("n_layers"))
    .filter(pl.col("n_layers") > 1)
)
print(f"Stocks present in more than one EqIndex: {overlap.height}")
if overlap.height == 0:
    print("No ticker overlap in this dataset -> inter-layer edges will be structurally empty.")
else:
    print(overlap)

In [ ]:
def build_multilayer_network(
    layer_graphs: dict[str, nx.Graph],
    eqindex_list: list[str],
    inter_layer_weight: float = 1.0,
) -> nx.Graph:
    """Node IDs: (stock, eq_index). Intra-layer edges come from each index's correlation
    graph. Inter-layer edges connect a stock across ALL pairs of layers it appears in --
    EqIndex has no natural adjacency order the way maturities do, so this is not
    restricted to 'consecutive' layers. Structurally empty for this dataset given the
    confirmed 0 cross-index ticker overlap (see the check above), but the code stays
    generic rather than hard-branching on that fact."""
    M = nx.Graph()
    for eq_index in eqindex_list:
        if eq_index not in layer_graphs:
            continue
        for u, v, data in layer_graphs[eq_index].edges(data=True):
            weight = 1.0 - data.get("weight", 0.5)
            M.add_edge((u, eq_index), (v, eq_index), weight=weight, layer="intra", eq_index=eq_index)
        for node in layer_graphs[eq_index].nodes():
            M.add_node((node, eq_index))

    stock_to_layers: dict[str, set] = {}
    for eq_index in eqindex_list:
        if eq_index in layer_graphs:
            for stock in layer_graphs[eq_index].nodes():
                stock_to_layers.setdefault(stock, set()).add(eq_index)

    for stock, layers_present in stock_to_layers.items():
        for layer_a, layer_b in itertools.combinations(sorted(layers_present), 2):
            M.add_edge((stock, layer_a), (stock, layer_b), weight=inter_layer_weight, layer="inter", stock=stock)

    return M

In [ ]:
multilayer_networks: list[nx.Graph] = []
for w_idx in window_indices:
    layer_graphs = {eq_index: layer_networks[eq_index][w_idx] for eq_index in eqindex_unique}
    M = build_multilayer_network(layer_graphs, eqindex_unique)
    multilayer_networks.append(M)

print(f"Built {len(multilayer_networks)} multiplex snapshots")

In [ ]:
for w_idx in [0, len(multilayer_networks) // 2, len(multilayer_networks) - 1]:
    M = multilayer_networks[w_idx]
    n_intra = sum(1 for _, _, d in M.edges(data=True) if d.get("layer") == "intra")
    n_inter = sum(1 for _, _, d in M.edges(data=True) if d.get("layer") == "inter")
    print(f"Window {w_idx} ({windows[w_idx][0]}..{windows[w_idx][1]}): "
          f"{M.number_of_nodes()} nodes, {n_intra} intra-edges, {n_inter} inter-edges")

## Chapter D: Multi-Layer Network Metrics

Per-layer and aggregate density, component count, and average degree across all
rolling-window snapshots.

In [ ]:
def compute_multilayer_metrics(M: nx.Graph, eqindex_list: list[str]) -> dict:
    """Per-layer + aggregate density/components/avg-degree for one multiplex snapshot."""
    per_layer = {}
    for eq_index in eqindex_list:
        layer_nodes = [n for n in M.nodes() if isinstance(n, tuple) and n[1] == eq_index]
        H = M.subgraph(layer_nodes)
        n_nodes, n_edges = H.number_of_nodes(), H.number_of_edges()
        per_layer[eq_index] = {
            "n_nodes": n_nodes,
            "n_edges": n_edges,
            "density": nx.density(H) if n_nodes > 1 else 0.0,
            "n_components": nx.number_connected_components(H) if n_nodes > 0 else 0,
            "avg_degree": (2 * n_edges / n_nodes) if n_nodes > 0 else 0.0,
        }
    n_intra = sum(1 for _, _, d in M.edges(data=True) if d.get("layer") == "intra")
    n_inter = sum(1 for _, _, d in M.edges(data=True) if d.get("layer") == "inter")
    aggregate = {
        "n_nodes_total": M.number_of_nodes(),
        "n_edges_total": M.number_of_edges(),
        "n_intra_edges": n_intra,
        "n_inter_edges": n_inter,
    }
    return {"per_layer": per_layer, "aggregate": aggregate}

In [ ]:
metrics_rows = []
for w_idx, M in enumerate(tqdm(multilayer_networks, desc="metrics")):
    metrics = compute_multilayer_metrics(M, eqindex_unique)
    date_start, date_end = windows[w_idx]
    for eq_index, layer_m in metrics["per_layer"].items():
        metrics_rows.append({
            "window_idx": w_idx, "date_start": date_start, "date_end": date_end,
            LAYER_COL: eq_index, **layer_m,
        })

metrics_df = pl.DataFrame(metrics_rows)
metrics_df.head(10)

In [ ]:
metrics_df.group_by(LAYER_COL).agg(
    pl.col("density").mean().alias("avg_density"),
    pl.col("n_edges").mean().alias("avg_n_edges"),
    pl.col("avg_degree").mean().alias("avg_avg_degree"),
).sort(LAYER_COL)

## Chapter E: Temporal Analysis of Multi-Layer Evolution

Visualize how network density and connectivity evolve across rolling windows, per
equity index, using both a static plotnine chart and interactive pyvis/3D views.

In [ ]:
evolution_plot = (
    ggplot(metrics_df.to_pandas(), aes(x="date_end", y="density", color=LAYER_COL))
    + geom_line(size=1)
    + labs(
        title="Multi-Layer Equity Network Density Over Time",
        x="Window End Date", y="Density", color="EqIndex",
    )
    + theme_minimal()
    + theme(figure_size=(12, 6))
)
evolution_plot

In [ ]:
evolution_plot.save("eq_multilayer_evolution.png", dpi=150)
print("Saved eq_multilayer_evolution.png")

In [ ]:
edges_over_time_plot = (
    ggplot(metrics_df.to_pandas(), aes(x="date_end", y="n_edges", color=LAYER_COL))
    + geom_line(size=1)
    + labs(title="Edge Count Over Time by Layer", x="Window End Date", y="Number of Edges", color="EqIndex")
    + theme_minimal()
    + theme(figure_size=(12, 6))
)
edges_over_time_plot

In [ ]:
# Interactive pyvis view of a representative window's multiplex network
from IPython.display import HTML, display

viz_window_idx = len(multilayer_networks) // 2
M_viz = multilayer_networks[viz_window_idx]
html = multilayer_graph_to_html(M_viz, title=f"Multi-Layer Equity Network (window {viz_window_idx})")
with open("eq_multilayer_pyvis.html", "w", encoding="utf-8") as f:
    f.write(html)
display(HTML(html))

In [ ]:
# 3D interactive-style view (matplotlib) stacking each EqIndex as a layer
layers_3d = []
for i, eq_index in enumerate(eqindex_unique):
    G_layer = layer_networks[eq_index][viz_window_idx]
    layers_3d.append(Layer(G_layer, z_pos=i, label=eq_index, node_size=30, node_color="#38bdf8"))

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection="3d")
lng = LayeredNetworkGraph(layers_3d, ax=ax)
lng.draw()
ax.set_title(f"3D Multi-Layer Equity Network (window {viz_window_idx})")
plt.tight_layout()
plt.savefig("eq_multilayer_3d.png", dpi=150)
plt.show()

## Chapter F: Inter-Layer Connectivity Analysis

Composition of intra- vs. inter-layer edges over time. Given the confirmed 0
cross-index ticker overlap (Chapter C), expect the composition to be ~100% intra-layer
for this dataset -- the code below degrades gracefully rather than branching on that.

In [ ]:
def count_edge_types(M: nx.Graph) -> dict:
    intra = sum(1 for _, _, d in M.edges(data=True) if d.get("layer") == "intra")
    inter = sum(1 for _, _, d in M.edges(data=True) if d.get("layer") == "inter")
    total = intra + inter
    return {
        "intra_edges": intra, "inter_edges": inter,
        "pct_intra": 100 * intra / total if total else 0.0,
        "pct_inter": 100 * inter / total if total else 0.0,
    }

edge_type_rows = []
for w_idx, M in enumerate(multilayer_networks):
    row = {"window_idx": w_idx, "date_end": windows[w_idx][1]}
    row.update(count_edge_types(M))
    edge_type_rows.append(row)

edge_type_df = pl.DataFrame(edge_type_rows)
edge_type_df.head(10)

In [ ]:
edge_type_plot = (
    ggplot(edge_type_df.to_pandas(), aes(x="date_end"))
    + geom_line(aes(y="intra_edges"), color="#0ea5e9", size=1)
    + geom_line(aes(y="inter_edges"), color="#a78bfa", size=1)
    + labs(title="Intra- vs. Inter-Layer Edge Counts Over Time",
           x="Window End Date", y="Edge Count")
    + theme_minimal()
    + theme(figure_size=(12, 6))
)
edge_type_plot

In [ ]:
edge_type_plot.save("eq_multilayer_edge_types.png", dpi=150)
print("Saved eq_multilayer_edge_types.png")

## Chapter G: Community Detection in Multi-Layer Networks

Spectral embedding (Adjacency Spectral Embedding) + KMeans clustering on the
multiplex network to detect stock communities spanning the layer structure.

In [ ]:
def detect_multilayer_communities(M: nx.Graph, n_components: int = 2, n_clusters: int = 4) -> dict:
    """ASE + KMeans community detection on a multiplex nx.Graph."""
    nodes = list(M.nodes())
    if len(nodes) < n_clusters:
        return {"labels": {}, "nodes": nodes}
    adj = nx.to_numpy_array(M, nodelist=nodes, weight="weight")
    ase = AdjacencySpectralEmbed(n_components=n_components)
    embedding = ase.fit_transform(adj)
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=0)
    labels = km.fit_predict(embedding)
    return {"labels": dict(zip(nodes, labels)), "nodes": nodes}

In [ ]:
community_window_idx = len(multilayer_networks) // 2
result = detect_multilayer_communities(multilayer_networks[community_window_idx])
print(f"Detected communities for window {community_window_idx}: "
      f"{len(set(result['labels'].values()))} clusters over {len(result['nodes'])} (stock, index) nodes")

In [ ]:
df_communities = pl.DataFrame([
    {"stock": node[0], LAYER_COL: node[1], "community": int(label)}
    for node, label in result["labels"].items()
])
df_communities.head(15)

In [ ]:
# Show a handful of representative tickers per index
sample_stocks = (
    df_communities.group_by(LAYER_COL)
    .head(5)
    .sort([LAYER_COL, "community"])
)
community_plot = (
    ggplot(sample_stocks.to_pandas(), aes(x="stock", y=LAYER_COL, fill="factor(community)"))
    + geom_tile()
    + labs(title="Sample Community Assignments by Stock and Index", x="Stock", y="EqIndex", fill="Community")
    + theme_minimal()
    + theme(figure_size=(10, 5), axis_text_x=element_text(rotation=90))
)
community_plot

# Part II: Factor Models, Residuals, and Stress

Nelson-Siegel has no equity analogue -- there is no ordered "maturity axis" across
CAC40/DAX30/FTSE100/HSI50/MIB40. Following the chosen adaptation, this section
replaces Nelson-Siegel curve-fitting with a **statistical PCA factor model per
EqIndex** (standard equity practice: PC1 approximates the index's common/market
factor, PC2/PC3 approximate style or sector factors) and builds residual networks
from PCA residuals instead of Nelson-Siegel residuals. `fit_pca` from
`tgraphportfolio.analysis.yield_curve_factors` is generic (operates on any
`(n_observations, n_series)` matrix) and is reused directly rather than reimplemented.

## Chapter H: Statistical Factor Extraction (PCA per EqIndex)

Fit a PCA factor model to each index's stock-return matrix (dates x stocks).

In [ ]:
N_COMPONENTS = 3
index_pca: dict[str, dict] = {}

for eq_index in eqindex_unique:
    df_wide = pivot_to_wide(
        df_clean.filter(pl.col(LAYER_COL) == eq_index),
        date_column=DATE_COL, name_column=NAME_COL, value_column=VALUE_COL,
    )
    node_cols = [c for c in df_wide.columns if c != DATE_COL]
    X = np.nan_to_num(df_wide.select(node_cols).to_numpy(), nan=0.0)
    if X.shape[1] < N_COMPONENTS + 1:
        continue
    pca_result = fit_pca(X, n_components=N_COMPONENTS)
    index_pca[eq_index] = {"pca": pca_result, "columns": node_cols, "dates": df_wide[DATE_COL].to_list()}

for eq_index, res in index_pca.items():
    ve = res["pca"].variance_explained
    print(f"{eq_index}: PC1-{len(ve)} variance explained = {np.round(ve, 3)}")

In [ ]:
variance_rows = []
for eq_index, res in index_pca.items():
    for pc_idx, ve in enumerate(res["pca"].variance_explained, start=1):
        variance_rows.append({LAYER_COL: eq_index, "component": f"PC{pc_idx}", "variance_explained": ve})
variance_df = pl.DataFrame(variance_rows)

variance_plot = (
    ggplot(variance_df.to_pandas(), aes(x="component", y="variance_explained", fill=LAYER_COL))
    + geom_col(position="dodge")
    + labs(title="PCA Variance Explained by Component and Index",
           x="Principal Component", y="Variance Explained", fill="EqIndex")
    + theme_minimal()
)
variance_plot

## Chapter I: PCA-Residual Networks

Correlation networks of PCA residuals (idiosyncratic, stock-specific co-movement
after removing the common/market factor) per equity index.

In [ ]:
def build_pca_residual_network(res: dict, threshold: float = 0.3) -> nx.Graph:
    """Correlation network of PCA residuals for one EqIndex."""
    residuals = res["pca"].residuals  # (n_dates, n_stocks), standardized residual space
    columns = res["columns"]
    residual_df = pl.DataFrame(residuals, schema=columns)
    measure_df = compute_measure("spearman_correlation", residual_df, columns, progress=None)
    return build_corr_nx(measure_df, independent_threshold=threshold)

residual_networks: dict[str, nx.Graph] = {
    eq_index: build_pca_residual_network(res) for eq_index, res in index_pca.items()
}

for eq_index, G in residual_networks.items():
    print(f"{eq_index}: residual network with {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Chapter J: Factor Evolution & Regime Classification

Rolling PC1 (market-factor) trajectory per index, and Gaussian-mixture regime
classification (e.g. calm / stressed / trending) in place of the yield curve's
normal / inverted / steepening regimes.

In [ ]:
FACTOR_WINDOW = 60   # ~3 trading months
FACTOR_STEP = 10

factor_trajectories: dict[str, pl.DataFrame] = {}
for eq_index, res in index_pca.items():
    pc1_scores = res["pca"].scores[:, 0]
    dates = res["dates"]
    rows = []
    for i in range(0, len(pc1_scores) - FACTOR_WINDOW + 1, FACTOR_STEP):
        window = pc1_scores[i:i + FACTOR_WINDOW]
        rows.append({
            "window_start": dates[i], "window_end": dates[i + FACTOR_WINDOW - 1],
            "pc1_mean": float(np.mean(window)), "pc1_std": float(np.std(window)),
        })
    factor_trajectories[eq_index] = pl.DataFrame(rows)

factor_trajectories[eqindex_unique[0]].head(10)

In [ ]:
regime_results: dict[str, dict] = {}
for eq_index, traj_df in factor_trajectories.items():
    factors = traj_df.select(["pc1_mean", "pc1_std"]).to_numpy()
    if len(factors) < 3:
        continue
    regime = classify_yield_curve_regimes(factors, n_regimes=3)
    regime_results[eq_index] = {"trajectory": traj_df, "regime": regime}
    print(f"{eq_index}: regimes = {regime.regime_names}")

In [ ]:
regime_plot_rows = []
for eq_index, r in regime_results.items():
    traj = r["trajectory"]
    labels = r["regime"].regime_labels
    for i in range(traj.height):
        regime_plot_rows.append({
            LAYER_COL: eq_index,
            "window_end": traj["window_end"][i],
            "pc1_mean": traj["pc1_mean"][i],
            "regime": r["regime"].regime_names[labels[i]],
        })
regime_plot_df = pl.DataFrame(regime_plot_rows)

regime_plot = (
    ggplot(regime_plot_df.to_pandas(), aes(x="window_end", y="pc1_mean", color="regime"))
    + geom_point(size=2)
    + facet_wrap("~" + LAYER_COL, scales="free_y")
    + labs(title="PC1 (Market-Factor) Trajectory & Regime by Index", x="Window End", y="PC1 mean")
    + theme_minimal()
    + theme(figure_size=(14, 8))
)
regime_plot

## Chapter K: Stress Testing & Correlation Breakdown

Rolling correlation stability -- average absolute correlation, correlation variance,
and count of significant edges over time. Directly applicable to equities as-is
(correlation breakdown during market stress, e.g. 2008, is a well-known equity
phenomenon and this window spans 2007-2019).

In [ ]:
def compute_correlation_stability(df: pl.DataFrame, eq_index: str, window_size: int = 60, step_size: int = 10) -> pl.DataFrame:
    """Rolling avg |correlation|, correlation variance, and significant-edge count for one index."""
    df_layer = df.filter(pl.col(LAYER_COL) == eq_index)
    df_wide = pivot_to_wide(df_layer, date_column=DATE_COL, name_column=NAME_COL, value_column=VALUE_COL)
    nodes = [c for c in df_wide.columns if c != DATE_COL]
    dates = df_wide[DATE_COL].to_list()

    rows = []
    for i in range(0, len(dates) - window_size + 1, step_size):
        sub = df_wide.slice(i, window_size).select(nodes)
        measure_df = compute_measure("spearman_correlation", sub, nodes, progress=None)
        corr = measure_df.to_numpy()
        iu = np.triu_indices_from(corr, k=1)
        vals = np.abs(corr[iu])
        rows.append({
            "window_start": dates[i], "window_end": dates[i + window_size - 1],
            "avg_abs_correlation": float(np.mean(vals)) if len(vals) else 0.0,
            "correlation_variance": float(np.var(vals)) if len(vals) else 0.0,
            "n_significant_edges": int(np.sum(vals >= 0.33)),
        })
    return pl.DataFrame(rows)

stability_dfs = {eq_index: compute_correlation_stability(df_clean, eq_index) for eq_index in eqindex_unique}
stability_dfs[eqindex_unique[0]].head(10)

In [ ]:
stability_plot_df = pl.concat([
    df.with_columns(pl.lit(eq_index).alias(LAYER_COL)) for eq_index, df in stability_dfs.items()
])

stress_plot = (
    ggplot(stability_plot_df.to_pandas(), aes(x="window_end", y="avg_abs_correlation", color=LAYER_COL))
    + geom_line(size=1)
    + labs(title="Rolling Average |Correlation| by Index (Correlation Breakdown Indicator)",
           x="Window End", y="Avg |Correlation|", color="EqIndex")
    + theme_minimal()
    + theme(figure_size=(12, 6))
)
stress_plot

## Chapter L: Advanced Centrality & Network Influence

Multi-layer degree centrality, summed across each index's residual network and
normalized by the number of layers a stock appears in.

In [ ]:
multilayer_centrality: dict[str, float] = {}
stock_layer_count: dict[str, int] = {}

for eq_index, G in residual_networks.items():
    degrees = dict(G.degree())
    for stock, deg in degrees.items():
        multilayer_centrality[stock] = multilayer_centrality.get(stock, 0) + deg
        stock_layer_count[stock] = stock_layer_count.get(stock, 0) + 1

for stock in multilayer_centrality:
    multilayer_centrality[stock] /= max(stock_layer_count[stock], 1)

centrality_df = pl.DataFrame([
    {"stock": s, "multilayer_degree_centrality": c} for s, c in multilayer_centrality.items()
]).sort("multilayer_degree_centrality", descending=True)
centrality_df.head(15)

## Chapter M: 3D Interactive Visualization

3D layered view of the PCA-residual networks (idiosyncratic co-movement, one layer
per index), plus centrality heatmaps.

In [ ]:
layers_3d_residual = [
    Layer(G, z_pos=i, label=eq_index, node_size=30, node_color="#a78bfa")
    for i, (eq_index, G) in enumerate(residual_networks.items())
]

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection="3d")
lng = LayeredNetworkGraph(layers_3d_residual, ax=ax)
lng.draw()
ax.set_title("3D PCA-Residual Networks by Index")
plt.tight_layout()
plt.savefig("eq_multilayer_residual_3d.png", dpi=150)
plt.show()

In [ ]:
heatmap_rows = []
for eq_index, G in residual_networks.items():
    degree = dict(G.degree())
    betweenness = nx.betweenness_centrality(G) if G.number_of_edges() else {n: 0.0 for n in G.nodes()}
    try:
        eigenvector = nx.eigenvector_centrality(G, max_iter=1000) if G.number_of_edges() else {n: 0.0 for n in G.nodes()}
    except nx.PowerIterationFailedConvergence:
        eigenvector = {n: 0.0 for n in G.nodes()}
    for stock in G.nodes():
        heatmap_rows.append({
            "stock": stock, LAYER_COL: eq_index,
            "degree": degree.get(stock, 0),
            "betweenness": betweenness.get(stock, 0.0),
            "eigenvector": eigenvector.get(stock, 0.0),
        })
heatmap_df = pl.DataFrame(heatmap_rows)

degree_heatmap = (
    ggplot(heatmap_df.to_pandas(), aes(x="stock", y=LAYER_COL, fill="degree"))
    + geom_tile()
    + scale_fill_gradient(low="#0f172a", high="#38bdf8")
    + labs(title="Residual-Network Degree by Stock and Index", x="Stock", y="EqIndex")
    + theme_minimal()
    + theme(figure_size=(14, 5), axis_text_x=element_text(rotation=90, size=5))
)
degree_heatmap

## Chapter N: Signal Subgraph Detection

Rank (stock-pair, index) combinations by how strongly their residual correlation
differs across the calm/stressed regimes identified in Chapter J -- these are the
edges that best discriminate between regimes.

In [ ]:
def compute_signal_edges(df: pl.DataFrame, eq_index: str, regime_labels: dict, top_k: int = 20) -> pl.DataFrame:
    """Rank residual-correlation edges by regime-discriminating power for one index.

    regime_labels: {window_end_date: regime_name} mapping from Chapter J.
    """
    df_layer = df.filter(pl.col(LAYER_COL) == eq_index)
    df_wide = pivot_to_wide(df_layer, date_column=DATE_COL, name_column=NAME_COL, value_column=VALUE_COL)
    nodes = [c for c in df_wide.columns if c != DATE_COL]

    regime_names = sorted(set(regime_labels.values()))
    if len(regime_names) < 2:
        return pl.DataFrame()

    corr_by_regime = {}
    for regime_name in regime_names:
        regime_dates = {d for d, r in regime_labels.items() if r == regime_name}
        sub = df_wide.filter(pl.col(DATE_COL).is_in(list(regime_dates))).select(nodes)
        if sub.height < 5:
            continue
        measure_df = compute_measure("spearman_correlation", sub, nodes, progress=None)
        corr_by_regime[regime_name] = measure_df.to_numpy()

    if len(corr_by_regime) < 2:
        return pl.DataFrame()

    regime_pair = list(corr_by_regime.keys())[:2]
    diff = np.abs(corr_by_regime[regime_pair[0]] - corr_by_regime[regime_pair[1]])
    iu = np.triu_indices_from(diff, k=1)
    rows = [
        {"stock_a": nodes[i], "stock_b": nodes[j], "abs_diff": float(diff[i, j])}
        for i, j in zip(*iu)
    ]
    return pl.DataFrame(rows).sort("abs_diff", descending=True).head(top_k)

In [ ]:
signal_edges_by_index: dict[str, pl.DataFrame] = {}
for eq_index, r in regime_results.items():
    traj, regime = r["trajectory"], r["regime"]
    labels_map = {
        traj["window_end"][i]: regime.regime_names[regime.regime_labels[i]]
        for i in range(traj.height)
    }
    signal_df = compute_signal_edges(df_clean, eq_index, labels_map)
    if signal_df.height:
        signal_edges_by_index[eq_index] = signal_df

if signal_edges_by_index:
    first_index = next(iter(signal_edges_by_index))
    print(f"Top signal edges for {first_index}:")
    signal_edges_by_index[first_index]

## Chapter O: Multi-Layer Community Detection with Layer Awareness

Per-layer ASE + KMeans clustering on the residual networks, with stable community IDs
compared across indices.

In [ ]:
def detect_layer_aware_communities(residual_networks: dict[str, nx.Graph], n_clusters: int = 3) -> dict[str, dict]:
    """ASE + KMeans clustering applied independently within each index's residual network."""
    results = {}
    for eq_index, G in residual_networks.items():
        nodes = list(G.nodes())
        if len(nodes) < n_clusters:
            continue
        adj = nx.to_numpy_array(G, nodelist=nodes, weight="weight")
        ase = AdjacencySpectralEmbed(n_components=2)
        embedding = ase.fit_transform(adj)
        km = KMeans(n_clusters=n_clusters, n_init=10, random_state=0)
        labels = km.fit_predict(embedding)
        results[eq_index] = dict(zip(nodes, labels))
    return results

layer_communities = detect_layer_aware_communities(residual_networks)
for eq_index, labels in layer_communities.items():
    print(f"{eq_index}: {len(set(labels.values()))} communities over {len(labels)} stocks")

In [ ]:
community_rows = [
    {"stock": stock, LAYER_COL: eq_index, "community": int(label)}
    for eq_index, labels in layer_communities.items()
    for stock, label in labels.items()
]
df_layer_communities = pl.DataFrame(community_rows)

sample_communities = df_layer_communities.group_by(LAYER_COL).head(8).sort([LAYER_COL, "community"])
community_heatmap = (
    ggplot(sample_communities.to_pandas(), aes(x="stock", y=LAYER_COL, fill="factor(community)"))
    + geom_tile()
    + labs(title="Layer-Aware Community Assignments (sample)", x="Stock", y="EqIndex", fill="Community")
    + theme_minimal()
    + theme(figure_size=(10, 5), axis_text_x=element_text(rotation=90))
)
community_heatmap